# StudySphere — Colab Training Runbook

Repo: `https://github.com/0xfrq/studysphere`

Notebook ini menjalankan urutan:

```text
GPU check -> mount Drive -> clone repo -> install -> login HF
-> generate dataset -> smoke test -> train planner -> train responder
-> merge adapter -> instruksi export/eval/integrasi
```

## Estimasi kasar di Colab T4

Dengan dataset sekarang (100.000 train per peran), 2 epoch, dan average sequence
sekitar 500 token:

| Model | Estimasi kasar | Catatan |
|---|---:|---|
| Planner Gemma 270M | 2–5 jam | Paling mungkin selesai dalam satu sesi |
| Responder Qwen3 1.7B QLoRA | 7–12 jam | Bisa menyentuh/melewati batas sesi gratis |
| Planner + responder + export/eval | > 12 jam | **Jangan digabung dalam satu sesi gratis** |

Angka ini bukan jaminan: T4 throttling, Drive I/O, evaluasi, dan checkpoint
menambah waktu. Karena itu notebook ini memakai checkpoint ke Google Drive
setiap `SAVE_STEPS` step dan bisa resume otomatis.

> **Penting:** repo GitHub harus berisi folder `ai/`. Kalau repo masih kosong,
> push dulu isi repo (tanpa `ai/data`) sebelum menjalankan notebook ini.


## 0. Runtime

Di Colab: **Runtime -> Change runtime type -> T4 GPU**.

Jalankan sel berikut. Kalau `torch.cuda.is_available()` bernilai `False`,
berhenti dan ganti runtime dulu.


In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("bf16:", torch.cuda.is_bf16_supported())


## 1. Mount Google Drive

Checkpoint dan adapter akan ditulis ke Drive supaya tidak hilang saat runtime
Colab berakhir.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Clone / update repo

Notebook mendukung dua layout:

- repo root langsung berisi `ai/`
- repo root berisi `fileconfig/ai/`

Kalau repo masih kosong, sel ini akan berhenti dengan pesan yang jelas.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/0xfrq/studysphere.git"
REPO_DIR = Path("/content/studysphere")

if not REPO_DIR.exists():
    result = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise SystemExit("git clone gagal")
elif (REPO_DIR / ".git").exists():
    # Jangan gagal kalau repo masih kosong / belum ada branch.
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        text=True,
        check=False,
    )

PROJECT_DIR = None
for candidate in (REPO_DIR, REPO_DIR / "fileconfig"):
    if (candidate / "ai").is_dir():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise SystemExit(
        "Folder ai/ tidak ditemukan. Push code ke "
        "https://github.com/0xfrq/studysphere dulu, "
        "atau upload zip code ke /content/studysphere."
    )

os.chdir(PROJECT_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("Isi:", sorted(p.name for p in PROJECT_DIR.iterdir())[:20])


## 3. Install dependency

Kalau Colab meminta restart runtime setelah install, restart lalu jalankan ulang
sel dari awal. Setelah restart, mount Drive lagi.


In [ ]:
!pip install -q -r ai/train/requirements.txt

import transformers
import peft
import datasets
import bitsandbytes
import torch

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("cuda:", torch.cuda.is_available())


## 4. Login Hugging Face

Gemma 3 biasanya gated. Buka halaman model di Hugging Face, setujui license,
lalu login dengan token yang punya akses.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 5. Generate dataset

Dataset tidak di-commit karena berukuran ~525 MB. Generator deterministik, jadi
cukup dibuat ulang di Colab. Ini juga membuat notebook bisa dipakai setelah
runtime reset.


In [ ]:
import subprocess

commands = [
    ["python", "ai/synth/generate_planner.py"],
    ["python", "ai/synth/generate_responder.py"],
    ["python", "ai/synth/generate_hard.py"],
    ["python", "ai/synth/validate.py"],
]

for cmd in commands:
    print("\nRUN:", " ".join(cmd))
    subprocess.run(cmd, check=True)


## 6. Konfigurasi training

Semua output training disimpan di Drive. `RESUME="auto"` akan mencari checkpoint
terbaru yang lengkap dan melanjutkan training.

- `SAVE_STEPS=200`: checkpoint setiap 200 optimizer step.
- `save-total-limit=3`: simpan maksimal 3 checkpoint agar Drive tidak penuh.
- Kalau ingin mulai dari nol: ubah `RESUME="none"` atau hapus folder output.


In [ ]:
from pathlib import Path
import os

RUN_ROOT = Path("/content/drive/MyDrive/studysphere_runs")
PLANNER_OUT = RUN_ROOT / "planner"
RESPONDER_OUT = RUN_ROOT / "responder"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PLANNER_OUT.mkdir(parents=True, exist_ok=True)
RESPONDER_OUT.mkdir(parents=True, exist_ok=True)

RESUME = "auto"          # "auto", "none", atau path checkpoint eksplisit
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 3

RUN_SMOKE = True         # set False kalau sudah pernah smoke test
USE_RESPONDER_SUBSET = False  # True = 25k train, lebih aman untuk Colab gratis

PLANNER_EPOCHS = 2
PLANNER_BATCH = 8
PLANNER_GRAD_ACCUM = 2
PLANNER_SEQ = 1024
PLANNER_GRAD_CKPT = False

RESPONDER_EPOCHS = 2
RESPONDER_BATCH = 4
RESPONDER_GRAD_ACCUM = 4
RESPONDER_SEQ = 1280
RESPONDER_GRAD_CKPT = False  # set True kalau OOM

print("RUN_ROOT:", RUN_ROOT)
print("RESUME:", RESUME)
print("SAVE_STEPS:", SAVE_STEPS)
print("USE_RESPONDER_SUBSET:", USE_RESPONDER_SUBSET)


## 7. Smoke test (opsional tapi disarankan)

Smoke test memakai 500 baris dan 1 epoch. Tujuannya memastikan chat template,
masking, dan QLoRA jalan sebelum membuang waktu di full training.

Kalau `RUN_SMOKE=False`, sel ini dilewati.


In [ ]:
import subprocess
from pathlib import Path

if not RUN_SMOKE:
    print("Smoke test dilewati.")
else:
    smoke_data = Path("ai/out/smoke_data")
    subprocess.run([
        "python", "ai/synth/generate_planner.py",
        "--out", str(smoke_data / "planner"),
        "--train", "500", "--val", "100", "--test", "100",
        "--variants", "20",
    ], check=True)
    subprocess.run([
        "python", "ai/synth/generate_responder.py",
        "--out", str(smoke_data / "responder"),
        "--train", "500", "--val", "100", "--test", "100",
        "--variants", "20",
    ], check=True)

    print("\nSmoke planner...")
    subprocess.run([
        "python", "ai/train/train_lora.py",
        "--base-model", "google/gemma-3-270m-it",
        "--train-file", str(smoke_data / "planner/train.jsonl"),
        "--val-file", str(smoke_data / "planner/val.jsonl"),
        "--output-dir", "ai/out/smoke-planner",
        "--epochs", "1",
        "--batch-size", "8",
        "--grad-accum", "2",
        "--lr", "1e-4",
        "--max-seq-length", "512",
        "--save-steps", "50",
        "--resume-from-checkpoint", "none",
    ], check=True)

    print("\nSmoke responder...")
    subprocess.run([
        "python", "ai/train/train_lora.py",
        "--base-model", "Qwen/Qwen3-1.7B",
        "--train-file", str(smoke_data / "responder/train.jsonl"),
        "--val-file", str(smoke_data / "responder/val.jsonl"),
        "--output-dir", "ai/out/smoke-responder",
        "--load-in-4bit",
        "--disable-thinking",
        "--epochs", "1",
        "--batch-size", "2",
        "--grad-accum", "4",
        "--lr", "1e-4",
        "--max-seq-length", "768",
        "--save-steps", "50",
        "--resume-from-checkpoint", "none",
    ], check=True)

    print("\nSmoke selesai.")
    print("planner adapter:", Path("ai/out/smoke-planner/adapter").exists())
    print("responder adapter:", Path("ai/out/smoke-responder/adapter").exists())


## 8. Train planner penuh — Gemma 3 270M

Output langsung ke Drive. Kalau runtime mati di tengah training, jalankan ulang
sel ini setelah setup + generate dataset; `RESUME="auto"` akan melanjutkan dari
checkpoint terakhir.

Jangan mengubah dataset, prompt, atau hyperparameter saat resume.


In [ ]:
import subprocess

planner_cmd = [
    "python", "ai/train/train_lora.py",
    "--base-model", "google/gemma-3-270m-it",
    "--train-file", "ai/data/planner/train.jsonl",
    "--val-file", "ai/data/planner/val.jsonl",
    "--output-dir", str(PLANNER_OUT),
    "--epochs", str(PLANNER_EPOCHS),
    "--batch-size", str(PLANNER_BATCH),
    "--grad-accum", str(PLANNER_GRAD_ACCUM),
    "--lr", "1e-4",
    "--max-seq-length", str(PLANNER_SEQ),
    "--save-steps", str(SAVE_STEPS),
    "--save-total-limit", str(SAVE_TOTAL_LIMIT),
    "--resume-from-checkpoint", RESUME,
]
if PLANNER_GRAD_CKPT:
    planner_cmd.append("--gradient-checkpointing")

print("RUN:", " ".join(planner_cmd))
subprocess.run(planner_cmd, check=True)
print("planner selesai:", PLANNER_OUT)


## 9. Train responder penuh — Qwen3 1.7B

Ini bagian yang paling mungkin melewati batas Colab gratis. Kalau waktu tidak
cukup, set `USE_RESPONDER_SUBSET=True` di sel konfigurasi untuk memakai 25.000
baris train.

Kalau OOM, ubah:

```python
RESPONDER_BATCH = 2
RESPONDER_GRAD_ACCUM = 8
RESPONDER_SEQ = 1024
RESPONDER_GRAD_CKPT = True
```


In [ ]:
import subprocess
from pathlib import Path

if USE_RESPONDER_SUBSET:
    subset = Path("ai/out/responder25k")
    subprocess.run([
        "python", "ai/synth/generate_responder.py",
        "--out", str(subset),
        "--train", "25000", "--val", "2500", "--test", "2500",
        "--variants", "250",
    ], check=True)
    responder_train = str(subset / "train.jsonl")
    responder_val = str(subset / "val.jsonl")
else:
    responder_train = "ai/data/responder/train.jsonl"
    responder_val = "ai/data/responder/val.jsonl"

responder_cmd = [
    "python", "ai/train/train_lora.py",
    "--base-model", "Qwen/Qwen3-1.7B",
    "--train-file", responder_train,
    "--val-file", responder_val,
    "--output-dir", str(RESPONDER_OUT),
    "--load-in-4bit",
    "--disable-thinking",
    "--epochs", str(RESPONDER_EPOCHS),
    "--batch-size", str(RESPONDER_BATCH),
    "--grad-accum", str(RESPONDER_GRAD_ACCUM),
    "--lr", "1e-4",
    "--max-seq-length", str(RESPONDER_SEQ),
    "--save-steps", str(SAVE_STEPS),
    "--save-total-limit", str(SAVE_TOTAL_LIMIT),
    "--resume-from-checkpoint", RESUME,
]
if RESPONDER_GRAD_CKPT:
    responder_cmd.append("--gradient-checkpointing")

print("RUN:", " ".join(responder_cmd))
subprocess.run(responder_cmd, check=True)
print("responder selesai:", RESPONDER_OUT)


## 10. Cek checkpoint

Kalau runtime mati, checkpoint terakhir ada di Drive. Sel ini hanya menampilkan
checkpoint yang lengkap.


In [ ]:
from pathlib import Path

def show_checkpoints(name, out_dir):
    checkpoints = []
    for path in Path(out_dir).glob("checkpoint-*"):
        if path.is_dir() and (path / "trainer_state.json").is_file():
            try:
                step = int(path.name.rsplit("-", 1)[1])
            except ValueError:
                continue
            checkpoints.append((step, path))
    checkpoints.sort()
    print(f"\n{name}: {out_dir}")
    if not checkpoints:
        print("  belum ada checkpoint")
    for step, path in checkpoints[-5:]:
        print(f"  step {step}: {path}")

show_checkpoints("planner", PLANNER_OUT)
show_checkpoints("responder", RESPONDER_OUT)


## 11. Merge adapter

Langkah ini bisa dijalankan di Colab. Untuk responder, merged model lebih besar
dan butuh RAM; kalau Colab kehabisan RAM, merge di laptop/GPU lain.


In [ ]:
import subprocess
from pathlib import Path

jobs = [
    (
        "google/gemma-3-270m-it",
        PLANNER_OUT / "adapter",
        PLANNER_OUT / "merged",
    ),
    (
        "Qwen/Qwen3-1.7B",
        RESPONDER_OUT / "adapter",
        RESPONDER_OUT / "merged",
    ),
]

for base_model, adapter, output in jobs:
    if not Path(adapter).is_dir():
        print("SKIP, adapter belum ada:", adapter)
        continue
    cmd = [
        "python", "ai/train/merge_adapter.py",
        "--base-model", base_model,
        "--adapter", str(adapter),
        "--output", str(output),
    ]
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("merged:", output)


## 12. Export GGUF dan Ollama

Bagian ini dijalankan di laptop setelah adapter/merged model diunduh, atau di
Colab kalau `llama.cpp` sudah tersedia. Urutan lengkap ada di `ai/README.md` §7.

Ringkasnya:

```sh
# planner
python llama.cpp/convert_hf_to_gguf.py ai/out/planner/merged \
  --outfile ai/out/planner/studysphere-planner-f16.gguf --outtype f16
llama-quantize ai/out/planner/studysphere-planner-f16.gguf \
  ai/out/planner/studysphere-planner-q4_k_m.gguf Q4_K_M

# responder
python llama.cpp/convert_hf_to_gguf.py ai/out/responder/merged \
  --outfile ai/out/responder/studysphere-responder-f16.gguf --outtype f16
llama-quantize ai/out/responder/studysphere-responder-f16.gguf \
  ai/out/responder/studysphere-responder-q4_k_m.gguf Q4_K_M
```

Letakkan GGUF di `ai/serve/ollama/`, lalu:

```sh
cd ai/serve/ollama
ollama create studysphere-planner -f Modelfile.planner
ollama create studysphere-responder -f Modelfile.responder
```


## 13. Evaluasi setelah GGUF/Ollama

Jalankan di laptop yang punya Ollama. Mulai dari 200 baris, lalu full test.

### Planner

```sh
python ai/eval/evaluate.py --task planner --data ai/data/planner/test.jsonl --base-url http://localhost:11434/v1 --model studysphere-planner --limit 200 --report ai/out/planner_predictions_200.jsonl

python ai/eval/evaluate.py --task planner --data ai/data/planner/test.jsonl --base-url http://localhost:11434/v1 --model studysphere-planner --report ai/out/planner_predictions.jsonl

python ai/eval/evaluate.py --task planner --data ai/data/eval/planner_hard.jsonl --base-url http://localhost:11434/v1 --model studysphere-planner --report ai/out/planner_hard_predictions.jsonl
```

### Responder

```sh
python ai/eval/evaluate.py --task responder --data ai/data/responder/test.jsonl --base-url http://localhost:11434/v1 --model studysphere-responder --limit 200 --report ai/out/responder_predictions_200.jsonl

python ai/eval/evaluate.py --task responder --data ai/data/responder/test.jsonl --base-url http://localhost:11434/v1 --model studysphere-responder --report ai/out/responder_predictions.jsonl

python ai/eval/evaluate.py --task responder --data ai/data/eval/responder_hard.jsonl --base-url http://localhost:11434/v1 --model studysphere-responder --report ai/out/responder_hard_predictions.jsonl
```

Jangan memilih checkpoint dari test set. Gunakan val loss untuk memilih epoch.


## 14. Integrasi desktop

Setelah evaluasi lolos:

1. Tahap A: responder-only untuk melihat model hidup.
2. Tahap B: planner + responder + MQTT + cache/budget.

Contoh environment di PowerShell:

```powershell
$env:STUDYSPHERE_LLM_URL="http://localhost:11434/v1"
$env:STUDYSPHERE_LLM_MODEL="studysphere-responder"
npm run tauri dev
```

Jangan memakai prompt desktop lama untuk menilai model SFT. Prompt inference
harus sama dengan prompt training; lihat `ai/docs/integration.md`.

Pitfall integrasi:

- `air_quality` station: level vs fresh/stale.
- `fan` station harus diterjemahkan ke boolean firmware.
- `vent` harus diterjemahkan ke `0/180`.
- Cache/budget node diterapkan di desktop.
- Command wajib divalidasi dengan whitelist.


## 15. Kalau runtime Colab mati

1. Buka Colab baru, pilih T4.
2. Mount Drive lagi.
3. Clone/pull repo dan install dependency.
4. Generate dataset lagi (deterministik).
5. Jalankan sel training yang sama dengan `RESUME="auto"`.

Checkpoint disimpan setiap `SAVE_STEPS` step, maksimal 3 checkpoint. Kalau ingin
mulai dari nol, set `RESUME="none"` atau hapus folder output di Drive.

Jangan mengubah dataset, prompt, atau hyperparameter di tengah resume.


## 16. Checklist

- [ ] `nvidia-smi` menunjukkan T4/L4.
- [ ] Drive ter-mount.
- [ ] Repo berisi `ai/`.
- [ ] Dependency terinstall.
- [ ] Hugging Face login dan Gemma license disetujui.
- [ ] Dataset 100k/10k/10k tervalidasi.
- [ ] Smoke test lulus.
- [ ] Planner selesai dan `run.json` ada.
- [ ] Responder selesai dan `run.json` ada.
- [ ] Merge/GGUF/Ollama berhasil.
- [ ] Evaluasi 200 baris dan full test dijalankan.
- [ ] Hard set dijalankan.
- [ ] Integrasi desktop diuji di node dan station.
- [ ] Baseline B0–B6 dijalankan.
- [ ] Test nyata 100–300 ucapan dikumpulkan.
